In [163]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim

In [164]:
from sklearn.datasets import fetch_california_housing

In [165]:
housing = fetch_california_housing()

In [166]:
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [167]:
X=housing.data

In [168]:
y=housing.target

In [169]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3,random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5,random_state=42)

In [170]:
scaler = StandardScaler()

In [171]:
X_train=scaler.fit_transform(X_train)
X_val=scaler.fit_transform(X_val)
X_test=scaler.fit_transform(X_test)

In [172]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [173]:
class LinearRegression(nn.Module):
    def __init__(self,input_dim):
        super(LinearRegression,self).__init__()
        self.linear=nn.Linear(input_dim,1)

    def forward(self,x):
        return self.linear(x)

In [174]:
model=LinearRegression(X_train.shape[1])

In [175]:
def train_model(model,criterion,optimiser,epochs):
    train_losses =[]
    for epoch in range(epochs):

        #train 
        model.train()

        #predict
        predictions = model(X_train_tensor)

        #loss
        loss = criterion(predictions,y_train_tensor)

        #grad
        optimiser.zero_grad()

        #backward pass
        loss.backward()

        #optimise
        optimiser.step()

        if (epoch+1)%100==0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {loss.item():.4f}')

In [176]:
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(),lr=0.01)
epochs=1000

In [177]:
train_model(model,criterion,optimiser,epochs)

Epoch [100/1000], Train Loss: 2.1087
Epoch [200/1000], Train Loss: 0.8839
Epoch [300/1000], Train Loss: 0.5795
Epoch [400/1000], Train Loss: 0.5296
Epoch [500/1000], Train Loss: 0.5240
Epoch [600/1000], Train Loss: 0.5234
Epoch [700/1000], Train Loss: 0.5234
Epoch [800/1000], Train Loss: 0.5234
Epoch [900/1000], Train Loss: 0.5234
Epoch [1000/1000], Train Loss: 0.5234


In [178]:
model.eval()

with torch.no_grad():
    predicted = model(X_test_tensor)
    print(f'MSE :{mean_squared_error(y_test, predicted)}')

MSE :0.49376080804992467


In [179]:
class MLP(nn.Module):
    def __init__(self,input_dim):
        super(MLP,self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return  self.model(x)

In [180]:
features=X_test_tensor.shape[1]
model = MLP(features)

In [181]:
def train_MLP(model,criterion,optimiser,epochs):
    train_losses =[]
    for epoch in range(epochs):

        #train 
        model.train()

        #predict
        predictions = model(X_train_tensor)

        #loss
        loss = criterion(predictions,y_train_tensor)

        #grad
        optimiser.zero_grad()

        #backward pass
        loss.backward()

        #optimise
        optimiser.step()

        if (epoch+1)%100==0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {loss.item():.4f}')

In [188]:
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(),lr=0.01)
epochs=1000
train_MLP(model,criterion,optimiser,epochs)

Epoch [100/1000], Train Loss: 0.2664
Epoch [200/1000], Train Loss: 0.2415
Epoch [300/1000], Train Loss: 0.2318
Epoch [400/1000], Train Loss: 0.2262
Epoch [500/1000], Train Loss: 0.2219
Epoch [600/1000], Train Loss: 0.2187
Epoch [700/1000], Train Loss: 0.2160
Epoch [800/1000], Train Loss: 0.2138
Epoch [900/1000], Train Loss: 0.2118
Epoch [1000/1000], Train Loss: 0.2104


In [191]:
from sklearn.metrics import mean_squared_error

model.eval()
with torch.no_grad():
    predicted = model(X_test_tensor)

    # Safely convert tensor to NumPy
    predicted_np = predicted.detach().numpy()
    y_test_np = y_test_tensor.detach().numpy()

    print(f'Test MSE: {mean_squared_error(y_test_np, predicted_np):.4f}')


Test MSE: 16.9349
